In [0]:
print("Hello Data Engineering")

Hello Data Engineering


In [0]:
storage_account = "ecomstoragemanas01"
storage_key = "*******************************************************"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
# display(
#  dbutils.fs.ls(
#     "abfss://landing@ecomstoragemanas01.dfs.core.windows.net/"
# )
#)

# Landing Layer

Read raw CSV files from Azure Data Lake Storage Gen2 (Landing Container).
No transformation is performed in this layer.
The objective is to preserve raw data exactly as received.


In [0]:
from pyspark.sql.functions import *

landing_path = "abfss://landing@ecomstoragemanas01.dfs.core.windows.net/"

In [0]:
customers_df = spark.read \
    .option("header", True) \
    .csv(f"{landing_path}customers.csv")

orders_df = spark.read \
    .option("header", True) \
    .csv(f"{landing_path}orders.csv")

order_items_df = spark.read \
    .option("header", True) \
    .csv(f"{landing_path}order_items.csv")

inventory_df = spark.read \
    .option("header", True) \
    .csv(f"{landing_path}inventory.csv")

In [0]:
# display(customers_df)

In [0]:
# display(orders_df)

In [0]:
# display(order_items_df)

In [0]:
# display(inventory_df)

# Bronze Layer

Store raw data as Delta Tables.
No cleaning or transformations are performed.
Metadata columns are added for audit purposes.

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

In [0]:
bronze_customers = customers_df \
.withColumn("bronze_ingestion_timestamp", current_timestamp()) \
.withColumn("source_file", input_file_name())

In [0]:
bronze_orders = orders_df \
.withColumn("bronze_ingestion_timestamp", current_timestamp()) \
.withColumn("source_file", input_file_name())

In [0]:
bronze_order_items = order_items_df \
.withColumn("bronze_ingestion_timestamp", current_timestamp()) \
.withColumn("source_file", input_file_name())

In [0]:
bronze_inventory = inventory_df \
.withColumn("bronze_ingestion_timestamp", current_timestamp()) \
.withColumn("source_file", input_file_name())

In [0]:
# display(bronze_customers)

# Bronze Database Creation

Create the Bronze database in the Hive Metastore to store raw data as Delta tables.  
This layer preserves the original data without applying any business transformations.  
The Bronze database acts as the foundation for the Medallion Architecture, enabling reliable data lineage, auditability, and downstream processing in the Silver and Gold layers.

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("USE bronze")

DataFrame[]

In [0]:
bronze_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.customers")

In [0]:
bronze_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.orders")

In [0]:
bronze_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.order_items")

In [0]:
bronze_inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.inventory")

In [0]:
spark.sql("SHOW TABLES IN bronze").show()

+--------+-----------+-----------+
|database|  tableName|isTemporary|
+--------+-----------+-----------+
|  bronze|  customers|      false|
|  bronze|  inventory|      false|
|  bronze|order_items|      false|
|  bronze|     orders|      false|
+--------+-----------+-----------+



# Verify Bronze Delta Tables

Verify that all Bronze Delta tables have been created successfully in the Hive Metastore.  
This step ensures that the raw datasets are stored correctly in Delta format along with ingestion metadata before proceeding to the Silver layer.

In [0]:
# display(spark.table("bronze.customers"))

In [0]:
# display(spark.table("bronze.orders"))

In [0]:
# display(spark.table("bronze.order_items"))

In [0]:
# display(spark.table("bronze.inventory"))

# Silver Layer

The Silver layer transforms raw Bronze data into clean, standardized, and validated datasets.

In this layer:
- Data types are converted to their appropriate formats.
- Missing and invalid records are identified.
- Duplicate records are removed.
- Data quality rules are applied.
- Invalid records are moved to quarantine tables.
- Clean datasets are prepared for analytical processing in the Gold layer.

# Create Silver Database

Create a dedicated database to store cleaned and validated Delta tables.  
The Silver database serves as the intermediate layer between raw Bronze data and business-ready Gold datasets.

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

# Read Bronze Delta Tables

Load the Bronze Delta tables into Spark DataFrames for data cleaning, validation, and transformation.

In [0]:
customers = spark.table("bronze.customers")
orders = spark.table("bronze.orders")
order_items = spark.table("bronze.order_items")
inventory = spark.table("bronze.inventory")

In [0]:
# display(customers)

# display(orders)

# display(order_items)

# display(inventory)

# Inspect Dataset Schema

Review the schema of each dataset before applying transformations. This helps identify incorrect data types that need to be converted in the Silver layer.

In [0]:
customers.printSchema()

orders.printSchema()

order_items.printSchema()

inventory.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- load_ts: string (nullable = true)
 |-- bronze_ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- load_ts: string (nullable = true)
 |-- bronze_ingestion_timestam

In [0]:
# customers.columns

In [0]:
# orders.columns

In [0]:
# order_items.columns

In [0]:
# inventory.columns

In [0]:
# display(customers)

In [0]:
# display(orders)

In [0]:
# display(order_items)

In [0]:
# display(inventory)

# Customer Data Type Standardization

Convert customer dataset columns to their appropriate data types before applying data quality rules. This ensures consistency, improves query performance, and prepares the dataset for downstream processing.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

customers_clean = customers \
.withColumn("signup_date", to_date("signup_date")) \
.withColumn("is_active", col("is_active").cast(BooleanType())) \
.withColumn("load_ts", to_timestamp("load_ts"))

In [0]:
customers_clean.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- load_ts: timestamp (nullable = true)
 |-- bronze_ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



# Customer Data Quality Validation

Validate customer records by checking mandatory fields and identifying invalid records. Valid records will continue to the Silver layer, while invalid records will be stored in a quarantine dataset for further review.

In [0]:
customers_valid = customers_clean.filter(
    col("customer_id").isNotNull() &
    col("email").isNotNull()
)

customers_quarantine = customers_clean.filter(
    col("customer_id").isNull() |
    col("email").isNull()
)

In [0]:
customers_valid = customers_valid.dropDuplicates(["customer_id"])

In [0]:
print("Valid Customers :", customers_valid.count())

print("Invalid Customers :", customers_quarantine.count())

# display(customers_valid)

# display(customers_quarantine)

Valid Customers : 10000
Invalid Customers : 0


# Orders Data Type Standardization

Convert the Orders dataset into appropriate data types before applying validation rules. Business identifiers are preserved as strings, while dates, timestamps, and numeric fields are converted to their respective data types.

In [0]:
orders_clean = orders \
.withColumn("order_date", to_timestamp("order_date")) \
.withColumn("total_amount", col("total_amount").cast("double")) \
.withColumn("discount_amount", col("discount_amount").cast("double")) \
.withColumn("load_ts", to_timestamp("load_ts"))

# Order Items Data Type Standardization

Standardize numeric and timestamp columns while preserving business identifiers as strings.

In [0]:
order_items_clean = order_items \
.withColumn("quantity", col("quantity").cast("int")) \
.withColumn("unit_price", col("unit_price").cast("double")) \
.withColumn("line_total", col("line_total").cast("double")) \
.withColumn("load_ts", to_timestamp("load_ts"))

# Inventory Data Type Standardization

Convert inventory metrics into appropriate numeric types and standardize timestamp columns for consistent downstream processing.

In [0]:
inventory_clean = inventory \
.withColumn("stock_quantity", col("stock_quantity").cast("int")) \
.withColumn("reorder_level", col("reorder_level").cast("int")) \
.withColumn("unit_cost", col("unit_cost").cast("double")) \
.withColumn("last_updated", to_timestamp("last_updated")) \
.withColumn("load_ts", to_timestamp("load_ts"))

In [0]:
customers_clean.printSchema()

orders_clean.printSchema()

order_items_clean.printSchema()

inventory_clean.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- load_ts: timestamp (nullable = true)
 |-- bronze_ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- load_ts: timestamp (nullable = true)
 |-- bronze_ingestion_

# Silver Data Quality Validation

Apply data quality rules to identify valid and invalid records.

Validation includes:
- Mandatory field checks
- Duplicate detection
- Data standardization
- Invalid record segregation into quarantine datasets

Only validated records will be promoted to the Silver layer.

In [0]:
customers_valid = customers_clean.filter(
    col("customer_id").isNotNull() &
    col("email").isNotNull()
)

customers_quarantine = customers_clean.filter(
    col("customer_id").isNull() |
    col("email").isNull()
)

customers_valid = customers_valid.dropDuplicates(["customer_id"])

In [0]:
orders_clean = orders_clean.withColumn(
    "status",
    lower(trim(col("status")))
)

orders_valid = orders_clean.filter(
    col("order_id").isNotNull() &
    col("customer_id").isNotNull() &
    col("total_amount").isNotNull()
)

orders_quarantine = orders_clean.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("total_amount").isNull()
)

orders_valid = orders_valid.dropDuplicates(["order_id"])

In [0]:
order_items_valid = order_items_clean.filter(
    col("item_id").isNotNull() &
    col("order_id").isNotNull() &
    (col("quantity") > 0)
)

order_items_quarantine = order_items_clean.filter(
    col("item_id").isNull() |
    col("order_id").isNull() |
    (col("quantity") <= 0)
)

order_items_valid = order_items_valid.dropDuplicates(["item_id"])

In [0]:
inventory_valid = inventory_clean.filter(
    col("sku_id").isNotNull() &
    (col("stock_quantity") >= 0)
)

inventory_quarantine = inventory_clean.filter(
    col("sku_id").isNull() |
    (col("stock_quantity") < 0)
)

inventory_valid = inventory_valid.dropDuplicates(["sku_id"])

# Validation Summary

Verify the number of valid and quarantined records after applying data quality rules.
This step confirms that only clean and standardized data will be promoted to the Silver layer.

In [0]:
print("Customers Valid:", customers_valid.count())
print("Customers Quarantine:", customers_quarantine.count())

print("Orders Valid:", orders_valid.count())
print("Orders Quarantine:", orders_quarantine.count())

print("Order Items Valid:", order_items_valid.count())
print("Order Items Quarantine:", order_items_quarantine.count())

print("Inventory Valid:", inventory_valid.count())
print("Inventory Quarantine:", inventory_quarantine.count())

Customers Valid: 10000
Customers Quarantine: 0
Orders Valid: 49500
Orders Quarantine: 501
Order Items Valid: 197014
Order Items Quarantine: 0
Inventory Valid: 4848
Inventory Quarantine: 0


# Store Silver Delta Tables

Store the validated and standardized datasets as Delta tables in the Silver database.

The Silver layer contains clean, deduplicated, and validated data that serves as the source for business analytics and Gold layer transformations.

In [0]:
customers_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.customers")

In [0]:
orders_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.orders")

In [0]:
order_items_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.order_items")

In [0]:
inventory_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.inventory")

# Store Quarantine Tables

Store all invalid records separately for auditing and future correction.
These datasets are excluded from downstream business processing.

In [0]:
customers_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.customers_quarantine")

In [0]:
orders_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.orders_quarantine")

In [0]:
order_items_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.order_items_quarantine")

In [0]:
inventory_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.inventory_quarantine")

# Verify Silver Layer

Verify that all Silver Delta tables and quarantine tables have been created successfully in the Hive Metastore.

In [0]:
spark.sql("SHOW TABLES IN silver").show(truncate=False)

+--------+----------------------+-----------+
|database|tableName             |isTemporary|
+--------+----------------------+-----------+
|silver  |customers             |false      |
|silver  |customers_quarantine  |false      |
|silver  |inventory             |false      |
|silver  |inventory_quarantine  |false      |
|silver  |order_items           |false      |
|silver  |order_items_quarantine|false      |
|silver  |orders                |false      |
|silver  |orders_quarantine     |false      |
+--------+----------------------+-----------+



In [0]:
# display(spark.table("silver.customers"))

# display(spark.table("silver.orders"))

# display(spark.table("silver.order_items"))

# display(spark.table("silver.inventory"))

# Gold Layer

The Gold layer contains business-ready datasets optimized for reporting, dashboards, and analytics.

This layer aggregates validated Silver data into meaningful business metrics that support decision-making and executive reporting.

# Create Gold Database

Create a dedicated database for storing business-ready analytical tables.

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

DataFrame[]

# Load Silver Tables

Load the cleaned Silver datasets to generate business KPIs and analytical reports.

In [0]:
customers = spark.table("silver.customers")
orders = spark.table("silver.orders")
order_items = spark.table("silver.order_items")
inventory = spark.table("silver.inventory")

# KPI 1 - Total Sales by Region

Calculate total sales generated from each region.

In [0]:
sales_by_region = orders.groupBy("region") \
.agg(sum("total_amount").alias("total_sales")) \
.orderBy(col("total_sales").desc())

# display(sales_by_region)

In [0]:
sales_by_region.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.sales_by_region")

# KPI 2 - Orders by Status

Count total orders for each order status.

In [0]:
orders_by_status = orders.groupBy("status") \
.count() \
.orderBy(col("count").desc())

# display(orders_by_status)

In [0]:
orders_by_status.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.orders_by_status")

# KPI 3 - Top Selling Products

Identify the highest-selling products based on total sales amount.

In [0]:
top_products = order_items.groupBy(
    "sku_id",
    "product_name"
).agg(
    sum("line_total").alias("total_sales")
).orderBy(
    col("total_sales").desc()
)

# display(top_products)

In [0]:
top_products.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.top_products")

# KPI 4 - Warehouse Inventory Summary

Calculate the total available stock across warehouses.

In [0]:
warehouse_inventory = inventory.groupBy("warehouse_id") \
.agg(
    sum("stock_quantity").alias("available_stock")
).orderBy(
    col("available_stock").desc()
)

# display(warehouse_inventory)

In [0]:
warehouse_inventory.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.warehouse_inventory")

# KPI 5 - Customer Revenue

Calculate the total revenue generated by each customer.

In [0]:
customer_revenue = orders.groupBy("customer_id") \
.agg(
    sum("total_amount").alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
)

# display(customer_revenue)

In [0]:
customer_revenue.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.customer_revenue")

# Verify Gold Layer

Verify that all Gold analytical tables have been successfully created in the Hive Metastore.

In [0]:
spark.sql("SHOW TABLES IN gold").show(truncate=False)

+--------+-------------------+-----------+
|database|tableName          |isTemporary|
+--------+-------------------+-----------+
|gold    |customer_revenue   |false      |
|gold    |orders_by_status   |false      |
|gold    |sales_by_region    |false      |
|gold    |top_products       |false      |
|gold    |warehouse_inventory|false      |
+--------+-------------------+-----------+



# Reconciliation Layer

The Reconciliation layer validates data consistency across the Landing, Bronze, Silver, and Gold layers.

It compares record counts and verifies that no unexpected data loss has occurred during the ETL process. This layer provides transparency, auditability, and confidence in the data pipeline.

# Create Reconciliation Database

Create a dedicated database to store audit and reconciliation reports.

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS reconciliation")

DataFrame[]

# Record Count Validation

Compare record counts between Bronze and Silver layers to verify successful processing.

In [0]:
from pyspark.sql import Row

reconciliation_data = [

    Row(table_name="customers",
        bronze_count=spark.table("bronze.customers").count(),
        silver_count=spark.table("silver.customers").count()),

    Row(table_name="orders",
        bronze_count=spark.table("bronze.orders").count(),
        silver_count=spark.table("silver.orders").count()),

    Row(table_name="order_items",
        bronze_count=spark.table("bronze.order_items").count(),
        silver_count=spark.table("silver.order_items").count()),

    Row(table_name="inventory",
        bronze_count=spark.table("bronze.inventory").count(),
        silver_count=spark.table("silver.inventory").count())

]

reconciliation_df = spark.createDataFrame(reconciliation_data)

# display(reconciliation_df)

In [0]:
reconciliation_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("reconciliation.record_count_validation")

In [0]:
display(
    spark.table("reconciliation.record_count_validation")
)

table_name,bronze_count,silver_count
order_items,200000,197014
customers,10000,10000
inventory,5000,4848
orders,50150,49500


In [0]:
print("ADF Pipeline Executed Successfully")

ADF Pipeline Executed Successfully
